# Property Follow-up CRM — Automated Build & Release Notebook (v1.1.0)

This Google Colab notebook automates the build pipeline for **Property Follow-up CRM**:
1. Mounts Google Drive (`MyDrive/PropertyCRM/`)
2. Clones the source repository into the Colab high-speed local filesystem (`/content/PropertyCRM_build/`)
3. Configures Android SDK command-line tools & accepts licenses
4. Runs unit tests (`testDebugUnitTest`)
5. Generates persistent release signing keystore on Drive (if not present)
6. Compiles Debug and Signed Release APKs (`assembleDebug assembleRelease`)
7. Computes SHA-256 checksums
8. Copies artifacts, test reports, and source archives to Google Drive

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os, subprocess, shutil, hashlib, datetime

drive.mount('/content/drive')

drive_base = '/content/drive/MyDrive/PropertyCRM'
dirs = ['SourceArchives', 'APKBuilds', 'TestReports', 'ReleaseNotes', 'DatabaseBackups']
for d in dirs:
    os.makedirs(os.path.join(drive_base, d), exist_ok=True)

print('Verified Google Drive Directory Structure:')
!ls -la /content/drive/MyDrive/PropertyCRM

In [ ]:
# Step 2: Set up Android SDK commandline tools
android_home = '/content/android-sdk'
os.environ['ANDROID_HOME'] = android_home
os.environ['ANDROID_SDK_ROOT'] = android_home

if not os.path.exists(os.path.join(android_home, 'cmdline-tools/latest/bin/sdkmanager')):
    print('Installing Android commandline tools...')
    os.makedirs(f'{android_home}/cmdline-tools', exist_ok=True)
    !curl -fsSL https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -o /content/cmdline-tools.zip
    !unzip -q /content/cmdline-tools.zip -d {android_home}/cmdline-tools/
    if os.path.exists(f'{android_home}/cmdline-tools/latest'):
        shutil.rmtree(f'{android_home}/cmdline-tools/latest')
    shutil.move(f'{android_home}/cmdline-tools/cmdline-tools', f'{android_home}/cmdline-tools/latest')
    if os.path.exists('/content/cmdline-tools.zip'):
        os.remove('/content/cmdline-tools.zip')

os.environ['PATH'] = f"{android_home}/cmdline-tools/latest/bin:{android_home}/platform-tools:{os.environ['PATH']}"

!yes | sdkmanager --licenses > /dev/null
!sdkmanager "platforms;android-34" "build-tools;34.0.0" > /dev/null
print('Android SDK Platform 34 ready!')

In [ ]:
# Step 3: Clone Project into Local Filesystem
build_dir = '/content/PropertyCRM_build'
if os.path.exists(build_dir):
    shutil.rmtree(build_dir)

repo_url = 'https://github.com/Vikash-45/PropertyCRM.git'
!git clone $repo_url $build_dir

os.chdir(build_dir)
!chmod +x gradlew
print('Repository cloned into:', build_dir)

In [ ]:
# Step 4: Run Unit Tests
%cd /content/PropertyCRM_build
!./gradlew testDebugUnitTest --stacktrace

In [ ]:
# Step 5: Configure Release Keystore and Build APKs
%cd /content/PropertyCRM_build
drive_keystore = '/content/drive/MyDrive/PropertyCRM/property_crm_release.keystore'
if not os.path.exists(drive_keystore):
    print('Generating persistent release keystore on Drive...')
    !keytool -genkeypair -v \
      -keystore {drive_keystore} \
      -alias propertycrm \
      -keyalg RSA \
      -keysize 2048 \
      -validity 10000 \
      -storepass "PropertyCRM@2026" \
      -keypass "PropertyCRM@2026" \
      -dname "CN=PropertyCRM, OU=Mobile, O=PropertyCRM, L=Noida, ST=UP, C=IN"

# Copy keystore into build root
shutil.copy2(drive_keystore, '/content/PropertyCRM_build/app/release.keystore')

os.environ['KEYSTORE_PATH'] = 'release.keystore'
os.environ['KEYSTORE_PASSWORD'] = 'PropertyCRM@2026'
os.environ['KEY_ALIAS'] = 'propertycrm'
os.environ['KEY_PASSWORD'] = 'PropertyCRM@2026'

!./gradlew assembleDebug assembleRelease --stacktrace

In [ ]:
# Step 6: Verify and Archive Artifacts to Drive
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
debug_apk = '/content/PropertyCRM_build/app/build/outputs/apk/debug/app-debug.apk'
release_apk = '/content/PropertyCRM_build/app/build/outputs/apk/release/app-release.apk'

assert os.path.exists(debug_apk), f"Debug APK missing: {debug_apk}"
assert os.path.exists(release_apk), f"Release APK missing: {release_apk}"

def get_sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

debug_sha = get_sha256(debug_apk)
release_sha = get_sha256(release_apk)

# Copy to Drive APKBuilds
dest_debug = f'/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_debug_{timestamp}.apk'
dest_release = f'/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_release_{timestamp}.apk'
shutil.copy2(debug_apk, dest_debug)
shutil.copy2(release_apk, dest_release)
shutil.copy2(debug_apk, '/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_debug_latest.apk')
shutil.copy2(release_apk, '/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_release_latest.apk')

# Checksum manifest
chk_file = f'/content/drive/MyDrive/PropertyCRM/APKBuilds/checksums_{timestamp}.txt'
with open(chk_file, 'w') as f:
    f.write(f"{debug_sha}  PropertyCRM_debug_{timestamp}.apk\n")
    f.write(f"{release_sha}  PropertyCRM_release_{timestamp}.apk\n")

# Archive Source
src_archive = f'/content/drive/MyDrive/PropertyCRM/SourceArchives/PropertyCRM_Source_v1.1.0_{timestamp}.tar.gz'
!tar --exclude='.git' --exclude='.gradle' --exclude='build' -czf {src_archive} -C /content/PropertyCRM_build .

# Test Reports
test_reports_src = '/content/PropertyCRM_build/app/build/reports/tests/testDebugUnitTest'
if os.path.exists(test_reports_src):
    shutil.copytree(test_reports_src, f'/content/drive/MyDrive/PropertyCRM/TestReports/test_report_{timestamp}', dirs_exist_ok=True)

print('='*60)
print('BUILD & ARCHIVE SUCCESSFUL!')
print('Debug APK SHA-256:', debug_sha)
print('Release APK SHA-256:', release_sha)
print('Release APK saved at:', dest_release)
print('='*60)